### ⚠️ ATENCIÓN: Ejecuta esta celda PRIMERO para instalar las librerías necesarias ⚠️

Si ves un error de `ModuleNotFoundError`, es porque esta celda no se ha ejecutado en este entorno. Por favor, ejecútala y luego reinicia el kernel si es necesario.

In [ ]:
import sys
!{sys.executable} -m pip install nltk joblib pandas scikit-learn numpy matplotlib seaborn

import nltk
nltk.download('stopwords')
nltk.download('wordnet')

print("✅ Instalación y descarga de NLTK completada.")

# Sistema de Detección de URLs de Spam

Este notebook implementa un sistema capaz de detectar automáticamente si una página web es spam o no basándose en su URL mediante una Máquina de Vectores de Soporte (SVM).

## Paso 1: Carga del Conjunto de Datos

In [ ]:
import pandas as pd

url = "https://breathecode.herokuapp.com/asset/internal-link?id=932&path=url_spam.csv"
df = pd.read_csv(url)

df.head()

## Paso 2: Preprocesamiento de los Enlaces

Limpiaremos las URLs dividiéndolas según los signos de puntuación, eliminando las palabras de parada (stopwords) y realizando la lematización.

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

def preprocess_url(url):
    url = re.sub(r'https?://', '', str(url))
    
    url = re.split(r'[^\w\s]', url)
    url = " ".join(url)
    
    words = url.lower().split()
    
    stop_words = set(stopwords.words('english'))
    words = [w for w in words if w not in stop_words]
    
    lemmatizer = WordNetLemmatizer()
    words = [lemmatizer.lemmatize(w) for w in words]
    
    return " ".join(words)

df['cleaned_url'] = df['url'].apply(preprocess_url)

df.head()

## Paso 3: Construcción de un SVM

Primero convertiremos el texto en vectores numéricos usando TF-IDF y luego entrenaremos un SVM con los parámetros por defecto.

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['cleaned_url'])
y = df['is_spam']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

svc = SVC()
svc.fit(X_train, y_train)

y_pred = svc.predict(X_test)
print(f"Precisión (Accuracy): {accuracy_score(y_test, y_pred)}")
print(classification_report(y_test, y_pred))

## Paso 4: Optimización del Modelo

Utilizaremos `GridSearchCV` para optimizar los hiperparámetros del SVM.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [1, 0.1, 0.01, 0.001],
    'kernel': ['linear', 'rbf']
}

grid = GridSearchCV(SVC(), param_grid, refit=True, verbose=2, cv=5)
grid.fit(X_train, y_train)

print(f"Mejores Parámetros: {grid.best_params_}")

y_grid_pred = grid.predict(X_test)
print(f"Precisión Optimizada: {accuracy_score(y_test, y_grid_pred)}")
print(classification_report(y_test, y_grid_pred))

## Paso 5: Guardado del Modelo

Guardaremos tanto el mejor modelo como el vectorizador para su uso futuro.

In [ ]:
import joblib
import os

model_dir = '../models/'
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

joblib.dump(grid.best_estimator_, os.path.join(model_dir, 'svm_model.pkl'))
joblib.dump(vectorizer, os.path.join(model_dir, 'tfidf_vectorizer.pkl'))

print(f"Modelo y vectorizador guardados en {model_dir}")